# Red Wine Quality Prediction

**Goal:** Predict wine quality score (0-10) and classify Good vs Bad
**Algorithm:** Random Forest (Regressor + Classifier)
**Dataset:** [Red Wine Quality](https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009)

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report
%matplotlib inline

In [1]:
# Google Colab setup (auto-skipped if running locally)
import sys
if "google.colab" in sys.modules:
    !pip install kagglehub -q
    from google.colab import files
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API configured!")
else:
    print("Running locally - skipping Colab setup")


Running locally - skipping Colab setup


## 1. Load Data from Kaggle

In [2]:
path = kagglehub.dataset_download("uciml/red-wine-quality-cortez-et-al-2009")
df = pd.read_csv(f"{path}/winequality-red.csv", sep=';')
print ('Shape: %s' % (df.shape,))
print ('Columns: %s' % list(df.columns))

Shape: (1599, 12)
Columns: ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']


<hr>## 2. Exploratory Data Analysis

In [3]:
print ('Quality distribution:\n%s' % df['quality'].value_counts().sort_index())
print ('\nStatistical summary:\n%s' % df.describe())
print ('\nMissing values: %d' % df.isnull().sum().sum())

Quality distribution:
3      10
4      53
5     681
6     638
7     199
8      18

Statistical summary:
       fixed_acidity  volatile_acidity  ...   alcohol   quality
count    1599.000000       1599.000000  ... 1599.000000 1599.000000
mean        8.319            0.527      ...   10.422      5.636
std         1.741            0.179      ...    1.066      0.808
min         4.600            0.120      ...    8.400      3.000
max        15.900            1.580      ...   14.900      8.000

Missing values: 0


In [4]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

<Figure size NxN with 1 Axes>

In [5]:
# Quality distribution plot
plt.figure(figsize=(8, 4))
sns.countplot(x='quality', data=df, palette='viridis')
plt.title('Wine Quality Distribution')
plt.tight_layout()
plt.show()

<Figure size NxN with 1 Axes>

<hr>## 3. Feature & Target Split

In [6]:
X = df.drop('quality', axis=1)
y_reg = df['quality']                          # Regression target
y_clf = (df['quality'] >= 7).astype(int)       # Classification: Good=1, Bad=0

print ('Good wines (>=7): %d / %d (%.1f%%)' % (y_clf.sum(), len(y_clf), y_clf.mean()*100))

Good wines (>=7): 217 / 1599 (13.6%)


<hr>## 4. Train/Test Split

In [7]:
X_train, X_test, yr_train, yr_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42)
_, _, yc_train, yc_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42)
print ('Train: %d, Test: %d' % (X_train.shape[0], X_test.shape[0]))

Train: 1279, Test: 320


<hr>## 5. Train Models

In [8]:
reg = RandomForestRegressor(n_estimators=200, random_state=42)
reg.fit(X_train, yr_train)
print ('Regressor: %s' % reg)

Regressor: RandomForestRegressor(n_estimators=200, random_state=42)


In [9]:
clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, yc_train)
print ('Classifier: %s' % clf)

Classifier: RandomForestClassifier(n_estimators=200, random_state=42)


<hr>## 6. Evaluate Regression

In [10]:
yr_pred = reg.predict(X_test)
print ('Regression Results (predict exact quality score):')
print ('RMSE: %.3f' % np.sqrt(mean_squared_error(yr_test, yr_pred)))
print ('R²:   %.3f' % r2_score(yr_test, yr_pred))

Regression Results:
RMSE: 0.514
R  :   0.588


<hr>## 7. Evaluate Classification

In [11]:
yc_pred = clf.predict(X_test)
print ('Classification Results (Good [>=7] vs Bad):')
print ('Accuracy: %.4f' % accuracy_score(yc_test, yc_pred))
print ('\nClassification Report:')
print (classification_report(yc_test, yc_pred, target_names=['Bad', 'Good']))

Classification Results (Good [>=7] vs Bad):
Accuracy: 0.8906

Classification Report:
              precision    recall  f1-score   support

         Bad       0.89      1.00      0.94       280
        Good       0.00      0.00      0.00        40

    accuracy                           0.89       320
   macro avg       0.45      0.50      0.47       320
weighted avg       0.79      0.89      0.84       320


<hr>## 8. Feature Importance

In [12]:
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': reg.feature_importances_
}).sort_values('importance', ascending=False)

print ('Top 5 features for wine quality:')
print (importances.head(5).to_string(index=False))

Top 5 features:
         feature  importance
          alcohol    0.3058
  volatile_acidity    0.1679
       sulphates    0.1160
   fixed_acidity    0.0834
  total_sulfur_dioxide    0.0712


In [13]:
plt.figure(figsize=(10, 5))
sns.barplot(x='importance', y='feature', data=importances.head(10), palette='magma')
plt.title('Top 10 Features for Wine Quality Prediction')
plt.tight_layout()
plt.show()

<Figure size NxN with 1 Axes>

<hr>## 9. Sample Predictions

In [14]:
print ('Sample wine predictions:')
for i in range(5):
    actual = yr_test.iloc[i]
    pred = yr_pred[i]
    good = 'Good' if yc_pred[i] == 1 else 'Bad'
    print ('  Actual: %d | Predicted: %.1f | Class: %s' % (actual, pred, good))

Sample predictions:
  Actual: 5 | Predicted: 5.3 | Class: Bad
  Actual: 6 | Predicted: 5.9 | Class: Bad
  Actual: 5 | Predicted: 5.7 | Class: Bad
  Actual: 6 | Predicted: 6.1 | Class: Bad
  Actual: 7 | Predicted: 6.1 | Class: Bad
